# 🧠 Semaine 2 — Jour 2 : Function Calling & Responses API

## Objectif du jour
- Comprendre le **Function Calling** d'OpenAI.
- Apprendre à définir un schéma **JSON** pour une fonction.
- Tester deux modes :
  1. **Mode mock (local)** : simulation sans appel API.
  2. **Mode réel (OpenAI API)** : avec une clé API valide.

## Références
- [OpenAI Function Calling Docs](https://platform.openai.com/docs/guides/function-calling)
- [OpenAI Responses API](https://platform.openai.com/docs/api-reference/responses)
- [JSON Schema Specification](https://json-schema.org/learn/getting-started-step-by-step.html)

---
💡 **Note** : Ce notebook t'entraîne à manipuler la *Responses API* via les fonctions, c'est une base essentielle avant de passer aux architectures d'agents complexes.

In [ ]:
# 🧩 Imports de base
import json
from typing import Dict, Any

# Si tu veux tester en mode OpenAI réel :
# !pip install openai
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

print('Environnement prêt ✅')

## 🧰 Étape 1 — Définir la fonction et son schéma JSON

On définit ici une fonction Python simple (conversion de température), accompagnée de sa **description JSON Schema** pour que le modèle puisse l’utiliser intelligemment.

In [ ]:
def convert_temperature(value: float, unit: str) -> Dict[str, Any]:
    """Convertit une température en Celsius, Fahrenheit ou Kelvin."""
    if unit == "C":
        return {"C": value, "F": value * 9/5 + 32, "K": value + 273.15}
    elif unit == "F":
        c = (value - 32) * 5/9
        return {"C": c, "F": value, "K": c + 273.15}
    elif unit == "K":
        c = value - 273.15
        return {"C": c, "F": c * 9/5 + 32, "K": value}
    else:
        raise ValueError(f"Unité non reconnue : {unit}")

function_schema = {
    "name": "convert_temperature",
    "description": "Convertit une température donnée en Celsius, Fahrenheit ou Kelvin.",
    "parameters": {
        "type": "object",
        "properties": {
            "value": {"type": "number", "description": "Valeur numérique de la température."},
            "unit": {"type": "string", "enum": ["C", "F", "K"], "description": "Unité actuelle de la température."}
        },
        "required": ["value", "unit"]
    }
}

print(json.dumps(function_schema, indent=2, ensure_ascii=False))

## 🧠 Étape 2 — Mode 1 : Mock local (sans API)

On simule ici la réponse d’un modèle qui comprend qu’il doit appeler la fonction.

In [ ]:
# Simulation d'une requête utilisateur
user_query = "Convertis 100 degrés Fahrenheit en Celsius"

# Mock : le modèle aurait décidé d'appeler la fonction
mock_response = {
    "function_call": {
        "name": "convert_temperature",
        "arguments": {"value": 100, "unit": "F"}
    }
}

args = mock_response["function_call"]["arguments"]
result = convert_temperature(**args)

print(f"Entrée : {args}\nRésultat : {result}")

## ⚙️ Étape 3 — Mode 2 : Exécution réelle avec OpenAI API

Tu peux exécuter ce code si tu disposes d’une clé API valide (variable d’environnement `OPENAI_API_KEY`).

In [ ]:
import os

if OpenAI and os.getenv("OPENAI_API_KEY"):
    client = OpenAI()

    messages = [
        {"role": "user", "content": "Convertis 25 degrés Celsius en Fahrenheit"}
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        functions=[function_schema],
        function_call="auto"
    )

    message = response.choices[0].message

    if message.function_call:
        args = json.loads(message.function_call.arguments)
        result = convert_temperature(**args)
        print(f"Résultat de la fonction appelée : {result}")
    else:
        print("Pas de function_call détecté.")
else:
    print("🔒 Mode mock uniquement — clé API non détectée.")

## 🧩 Étape 4 — Extension : plusieurs fonctions disponibles

Tu peux définir plusieurs fonctions et laisser le modèle choisir laquelle appeler.

C’est le principe de base de l’**agent orchestrateur**, qui choisit dynamiquement entre plusieurs outils.

In [ ]:
def add_numbers(a: float, b: float) -> float:
    return a + b

add_schema = {
    "name": "add_numbers",
    "description": "Additionne deux nombres.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number"},
            "b": {"type": "number"}
        },
        "required": ["a", "b"]
    }
}

available_functions = [function_schema, add_schema]

print("Deux fonctions prêtes à être appelées dynamiquement ✅")

## ✅ Bilan du jour
- Compris le rôle du Function Calling dans les agents IA.
- Savoir définir un schéma JSON propre.
- Testé deux modes d’exécution (mock et réel).
- Préparation à la prochaine étape : **construction d’un agent mono-outil.**